# 87 · Gene model audit: tradeSeq, the NB assumptions and the parameters

Three questions about the NB gene model of notebooks 65–85 (`standard_gene_model` and the positional F-test):
- **A.** Is it tradeSeq? Side by side from the code, then tradeSeq itself run on 71's units, genes and partitions.
- **B.** Do the tubule units fit the NB assumptions (zeros, variance, residuals, offset, constant dispersion)?
- **C.** Are the parameters (basis df, knots, weights, dispersion, offset) the best, and does the choice matter?

**Protocol.** `docs/results/pt-gene-model-audit.md` (frozen in 0c8d28a, before any fit). Human vs mouse PT on the
package's elastic curve, 71's inputs. Descriptive apart from the per-gene tests the protocol states.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from scipy.stats import chi2, false_discovery_control, spearmanr

EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: universe, sets, stage cache, statistics
output = results_root / 'pt_gene_model_audit'
tables, figures, ts_dir = output / 'tables', output / 'figures', output / 'tradeseq'
for folder in (tables, figures, ts_dir):
    folder.mkdir(parents=True, exist_ok=True)
R_SCRIPT = project / 'analysis' / 'scripts' / 'tradeseq_audit.R'
RSCRIPT = Path(os.environ.get('TRADESEQ_RSCRIPT', Path(sys.prefix).parent / 'tradeseq-audit' / 'bin' / 'Rscript'))
DS = 'human_vs_mouse'
REQUIRED = [EXPORT, upstream13 / 'ortholog_map_used.csv', R_SCRIPT, RSCRIPT,
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70 and 71; create the tradeseq-audit environment)')

NOTEBOOK_LOGIC_VERSION = '87.gene_model_audit.1'   # Bump when a cached calculation changes.
FINGERPRINT = 'efafc9ac2e99'
BASIS_DF, DF1 = 6, 6
DF2 = {'condition': 12, 'relabeling': 6}
CAMERA_COR, ALPHA, NCDR_MAX, NULL_RATE, CALIBRATION_MAX = .01, .05, .25, .05, .075   # 71's rule
PER_DECILE, SEED = 150, 87
MARKERS = ['Slc34a1', 'Slc5a2', 'Slc5a12', 'Slc22a6', 'Slc22a8', 'Slc22a7', 'Slc7a13', 'Lrp2', 'Cubn', 'Gpx3', 'Fxyd2',
           'Aqp1', 'Slc13a3', 'Cyp4b1', 'Hnf4a', 'Spp1', 'Umod']
DFS = (3, 4, 5, 6, 7, 8, 10, 12)
VARIANT_DFS = (4, 5, 8, 10)
N_FOLDS = 5
MISFIT = {'zero': .02, 'ks': .05, 'slope': .2, 'alpha_ratio': 2.}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
R_WORKERS = int(os.environ.get('TRADESEQ_WORKERS', '48'))

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
C_OURS, C_TS, C_TSU, C_RAW = '#0072B2', '#D55E00', '#E69F00', '#56B4E9'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 80, 'display.max_rows', 200)
GUARDS = []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only)."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION, '· Rscript', RSCRIPT)

## 2 · Inputs and guards (not cached)

G1: `hvm_inputs` reproduces 71's universe, units and sets; G2: 71's stored models are read for the species split and
both relabelings, with their denominators and gene statistics.

In [ ]:
from pseudospace import gene_model_audit as ga
from pseudospace import nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_calibration import contrast_designs
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f, partition_groups
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
UNITS = read_pt_units(EXPORT, 'hvm')
assert UNITS.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G1: universe differs from 71'
assert INP['gene_sets'] == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: sets differ'
COUNTS = INP['counts'].tocsc()
LIBRARY = np.asarray(INP['library'], float)
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
HUMAN = np.asarray(INP['case'], bool)
OBS = INP['obs']
N_SPOTS = OBS['n_spots'].to_numpy(float)
LOG_MEAN = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
SETS = INP['gene_sets']
PARTITIONS = partition_groups(SPECIMEN, HUMAN)
LABELS = list(PARTITIONS)
SHORT = dict(zip(LABELS, ['condition', 'relabeling 1', 'relabeling 2']))
slug = lambda label: label.replace(':', '_').replace('+', '_')               # noqa: E731
M71 = {label: stored(T71, f'gene_model_{DS}_{slug(label)}') for label in LABELS}
D71 = {label: stored(T71, f'specimen_shape_{DS}_{slug(label)}') for label in LABELS}
S71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
assert S71.index.astype(str).tolist() == GENES
KNOTS = M71['species']['knots']
GRID = M71['species']['grid']
np.testing.assert_allclose(KNOTS, gam_internal_knots(POSITION, BASIS_DF), rtol=0, atol=1e-12)
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
ALPHA71 = M71['species']['alpha']
GUARDS.append(f'G1: {len(GENES):,} genes, {len(POSITION):,} units and {len(SETS):,} sets equal 71; 71\'s knots '
              f'{np.round(KNOTS, 3).tolist()} and grid are recomputed exactly')
GUARDS.append('G2: 71\'s species and relabeling models, denominators and gene statistics read from its stage cache')
print('\n'.join('· ' + g for g in GUARDS))

rng = np.random.default_rng(SEED)
decile = pd.qcut(pd.Series(LOG_MEAN).rank(method='first'), 10, labels=False).to_numpy()
SUBSET = sorted({GENES[i] for k in range(10) for i in rng.choice(np.flatnonzero(decile == k), PER_DECILE, replace=False)}
                | {g for g in MARKERS if g in set(GENES)})
SUB = np.array([GENES.index(g) for g in SUBSET])
print(f'Gene subset G: {len(SUBSET):,} genes ({PER_DECILE} per decile of log mean count, plus '
      f'{len([g for g in MARKERS if g in set(GENES)])} markers)')

## 3 · A1 · The modelling choices, side by side

Read from tradeSeq 1.20.0's source (`.fitGAM`, `.findKnots`, `.get_offset`, `conditionTest`, `associationTest`) and
from ours (`nb_gam`, `standard_gene_model`, `specimen_shape_lr`, `moderated_f`).

In [ ]:
A1 = pd.DataFrame([
    ('Count model', 'NB, log link: mgcv::gam(family = "nb")', 'NB, log link: own IRLS (R glm convergence rule)', 'same'),
    ('Dispersion', 'one θ per gene inside the REML outer iteration; no shrinkage', 'one α per gene, ML under M_full, then fixed; no shrinkage', 'same idea; estimated differently'),
    ('Smoother', "cubic regression spline bs='cr', k = 6, knots at quantiles 0, .2, …, 1", 'cubic B-spline, df 6, interior knots at quartiles', 'both cubic, about 6 dimensions'),
    ('Penalty', 'second-derivative penalty, λ by REML, shared across conditions', 'none (fixed df)', 'different'),
    ('Conditions', 'one unconstrained smooth per condition', 'group main effect + group × basis', 'equivalent span'),
    ('Offset', 'log(library × TMM factor)', 'log(library over the measured orthologs, mt-free)', 'different normaliser'),
    ('Weights', 'none', 'each group half, each specimen an equal share', 'different'),
    ('Specimens', 'none unless given in U', 'specimen intercepts within group', 'different'),
    ('Condition test', 'Wald χ² of smooth differences at the knots (level + shape), Vp', 'LR of shape only (T_spatial) ÷ specimen-shape LR, limma-moderated F', 'different'),
    ('Association test', 'Wald χ²: smooth constant', 'LR: spline vs specimen intercepts', 'same question, different test'),
    ('Replication unit', 'the unit (cell)', 'the specimen (F denominator)', 'different'),
], columns=['choice', 'tradeSeq 1.20.0 defaults', 'ours (71)', 'verdict'])
display(A1)
A1.to_csv(tables / 'a1_side_by_side.csv', index=False)

## 4 · A2 · tradeSeq on 71's units, genes and partitions

The counts, positions and partitions are exported once; `tradeseq_audit.R` runs in the `tradeseq-audit`
environment. The run is skipped when its key (inputs and script) matches the stored one.

In [ ]:
from scipy.io import mmwrite

units_table = pd.DataFrame({'unit': OBS.index.astype(str), 'position': POSITION, 'specimen': SPECIMEN,
                            'species': np.where(HUMAN, 'human', 'mouse'), 'library': LIBRARY, 'n_spots': N_SPOTS})
for k, label in enumerate(LABELS[1:], start=1):
    units_table[f'relabel{k}'] = np.where(PARTITIONS[label][0] == 1, 'g1', 'g0')
TS_KEY = digest({'counts': digest(COUNTS), 'units': digest(units_table.to_numpy().astype(str)), 'genes': GENES,
                 'subset': SUBSET, 'grid': GRID, 'script': hashlib.sha256(R_SCRIPT.read_bytes()).hexdigest()})
key_file = ts_dir / 'key.json'
outputs = [f'condition_{a}_{b}.csv' for a in ('species', 'relabel1', 'relabel2') for b in ('default', 'U')] + \
          ['association_mouse.csv', 'gam_subset.csv', 'curves_subset.csv', 'offsets.csv', 'session.txt']
fresh = key_file.is_file() and json.loads(key_file.read_text()).get('key') == TS_KEY and all((ts_dir / f).is_file() for f in outputs)
if fresh:
    print('[tradeSeq] stored run matches', TS_KEY)
    TS_SECONDS = json.loads(key_file.read_text())['seconds']
else:
    mmwrite(ts_dir / 'counts.mtx', sparse.csr_matrix(COUNTS.T))
    (ts_dir / 'genes.txt').write_text('\n'.join(GENES) + '\n')
    (ts_dir / 'subset.txt').write_text('\n'.join(SUBSET) + '\n')
    (ts_dir / 'grid.txt').write_text('\n'.join(f'{x:.17g}' for x in GRID) + '\n')
    units_table.to_csv(ts_dir / 'units.csv', index=False)
    t0 = time.perf_counter()
    env = dict(os.environ, OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    run = subprocess.run([str(RSCRIPT), str(R_SCRIPT), str(ts_dir), str(R_WORKERS)], capture_output=True, text=True, env=env)
    print(run.stderr[-3000:])
    if run.returncode:
        raise RuntimeError('tradeSeq run failed')
    TS_SECONDS = time.perf_counter() - t0
    key_file.write_text(json.dumps({'key': TS_KEY, 'seconds': TS_SECONDS}))
    (ts_dir / 'counts.mtx').unlink()                       # 1 GB; the key keeps the digest
print(f'tradeSeq run: {TS_SECONDS / 60:.1f} min')
print((ts_dir / 'session.txt').read_text().splitlines()[-3:])

### A2a–c · Ranking, calibration and calls

Ours: raw T_total LR (level + shape, χ² 7 df), raw T_spatial LR (χ² 6 df) and the positional F-test (71's p). tradeSeq:
`conditionTest` p of TS-default and TS-U. Calibration: the share of genes with p ≤ 0.05 under each relabeling.

In [ ]:
def ts_table(name):
    t = pd.read_csv(ts_dir / f'{name}.csv').set_index('gene').reindex(GENES)
    return t


P = {}
for label in LABELS:
    short = SHORT[label]
    m = M71[label]
    ok = m['stat|converged'].astype(bool) & ~m['stat|separated'].astype(bool)
    P[('ours: T_total LR (χ², 7 df)', short)] = pd.Series(np.where(ok, chi2.sf(m['stat|T_total'], 7), np.nan), index=GENES)
    P[('ours: T_spatial LR (χ², 6 df)', short)] = pd.Series(np.where(ok, chi2.sf(m['stat|T_spatial'], 6), np.nan), index=GENES)
    P[('ours: positional F-test', short)] = S71[(short, 'p')].where(ok)
    tag = {'condition': 'species', 'relabeling 1': 'relabel1', 'relabeling 2': 'relabel2'}[short]
    for arm, name in (('default', 'tradeSeq conditionTest'), ('U', 'tradeSeq + specimen U')):
        t = ts_table(f'condition_{tag}_{arm}')
        # tradeSeq reports 1 - pchisq(W, df), which is exactly 0 below about 1e-16; the same Wald statistic's upper
        # tail is recomputed accurately (recorded deviation; calls and shares at 0.05 do not change).
        P[(name, short)] = pd.Series(chi2.sf(t.waldStat, t.df), index=t.index).where(t.converged.astype(bool) & t.waldStat.notna())
METHODS = list(dict.fromkeys(k[0] for k in P))
rows = []
for method in METHODS:
    pc = P[(method, 'condition')]
    q = pd.Series(np.nan, index=GENES)
    q[pc.notna()] = false_discovery_control(pc.dropna().to_numpy())
    calls = int((q <= ALPHA).sum())
    rel = [P[(method, r)] for r in ('relabeling 1', 'relabeling 2')]
    rel_calls = []
    for pr in rel:
        qr = false_discovery_control(pr.dropna().to_numpy())
        rel_calls.append(int((qr <= ALPHA).sum()))
    share = float(np.mean(np.concatenate([pr.dropna().to_numpy() for pr in rel]) <= ALPHA))
    rows.append({'method': method, 'genes tested': int(pc.notna().sum()), 'species calls (BH ≤ 0.05)': calls,
                 'relabeled calls': f'{rel_calls[0]}, {rel_calls[1]}', 'NCDR': np.mean(rel_calls) / calls if calls else np.nan,
                 'relabeled share p ≤ 0.05': share, 'calibrated (≤ 0.075)': share <= CALIBRATION_MAX})
A2BC = pd.DataFrame(rows).set_index('method')
display(A2BC.round(3))
A2BC.to_csv(tables / 'a2bc_calibration_and_calls.csv')

ref = -np.log10(P[('tradeSeq conditionTest', 'condition')].clip(lower=1e-300))
rows = []
for method in METHODS:
    other = -np.log10(P[(method, 'condition')].clip(lower=1e-300))
    both = ref.notna() & other.notna()
    top_a = set(ref[both].nlargest(int(.1 * both.sum())).index)
    top_b = set(other[both].nlargest(int(.1 * both.sum())).index)
    rows.append({'method': method, 'Spearman with tradeSeq conditionTest': spearmanr(ref[both], other[both]).statistic,
                 'top-10% overlap (Jaccard)': len(top_a & top_b) / len(top_a | top_b), 'genes': int(both.sum())})
A2A = pd.DataFrame(rows).set_index('method')
display(A2A.round(3))
A2A.to_csv(tables / 'a2a_ranking.csv')

### A2d · Within mouse: `associationTest` against our LR (spline against specimen intercepts)

In [ ]:
mouse = ~HUMAN
mcounts = COUNTS[np.flatnonzero(mouse)]
spec_m = SPECIMEN[mouse]
dummies = np.column_stack([(spec_m == s).astype(float) for s in sorted(set(spec_m))])
basis_m = make_gam_design(POSITION[mouse], KNOTS)[:, 1:]
designs_m = {'base': dummies, 'full': np.column_stack([dummies, basis_m])}


def within_mouse():
    off = np.log(LIBRARY[mouse])
    disp = nb_gam.nb_dispersion(mcounts, designs_m['full'], np.ones(mouse.sum()), off, n_jobs=N_JOBS)
    lr = nb_gam.nested_nb_lr(mcounts, designs_m, np.ones(mouse.sum()), off, disp, {'LR': ('base', 'full')}, n_jobs=N_JOBS)
    return {'LR': lr['LR'], 'converged': lr['converged'], 'separated': lr['separated'], 'alpha': disp['alpha']}


CODE = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (ga, nb_gam, pathway_calibration, pathway_pipelines, stats_gam)])
INPUT_KEY = {'counts': digest(COUNTS), 'library': LIBRARY, 'position': POSITION, 'specimen': SPECIMEN, 'genes': GENES}


def cached(stage, compute, **kw):
    return cached_payload(stage, compute, root=output / 'stage_cache', code=CODE, **kw)


WM = cached('within_mouse_lr', within_mouse, inputs={**INPUT_KEY, 'knots': KNOTS})
assoc = ts_table('association_mouse')
ours_m = pd.Series(np.where(WM['converged'] & ~WM['separated'], chi2.sf(WM['LR'], basis_m.shape[1]), np.nan), index=GENES)
ts_m = pd.Series(chi2.sf(assoc.waldStat, assoc.df), index=assoc.index).where(assoc.converged.astype(bool) & assoc.waldStat.notna())
both = ours_m.notna() & ts_m.notna()
a = -np.log10(ours_m[both].clip(lower=1e-300))
b = -np.log10(ts_m[both].clip(lower=1e-300))
n10 = int(.1 * both.sum())
A2D = pd.Series({'genes': int(both.sum()), 'Spearman of −log10 p': spearmanr(a, b).statistic,
                 'top-10% Jaccard': len(set(a.nlargest(n10).index) & set(b.nlargest(n10).index)) / len(set(a.nlargest(n10).index) | set(b.nlargest(n10).index)),
                 'ours BH ≤ 0.05': int((false_discovery_control(ours_m[both]) <= ALPHA).sum()),
                 'tradeSeq BH ≤ 0.05': int((false_discovery_control(ts_m[both]) <= ALPHA).sum())}, name='within mouse')
display(A2D.to_frame().round(3))
A2D.to_csv(tables / 'a2d_within_mouse.csv')

### A2e · Does the penalty change the fit? (gene subset G)

tradeSeq's REML-chosen edf per condition smooth (6 coefficients, so at most 6), and its species curves against 71's
group curves on 71's grid, both centred (shape only).

In [ ]:
GAM = pd.read_csv(ts_dir / 'gam_subset.csv').set_index('gene')
CURVES = pd.read_csv(ts_dir / 'curves_subset.csv')
edf = GAM[['edf_human', 'edf_mouse']]
print('edf per condition smooth (of 6):')
display(edf.describe().round(2))
rows = []
idx71 = {g: i for i, g in enumerate(GENES)}
for g in GAM.index:
    i = idx71[g]
    for cond, key in (('human', 'case'), ('mouse', 'reference')):
        ts_curve = CURVES[(CURVES.gene == g) & (CURVES.condition == cond)].sort_values('point').value.to_numpy()
        ours = M71['species'][key][i]
        if len(ts_curve) != len(GRID) or not np.isfinite(ours).all():
            continue
        a_, b_ = ts_curve - ts_curve.mean(), ours - ours.mean()
        rows.append({'gene': g, 'condition': cond, 'r': np.corrcoef(a_, b_)[0, 1] if a_.std() > 0 and b_.std() > 0 else np.nan,
                     'RMS difference (ln)': float(np.sqrt(np.mean((a_ - b_) ** 2))), 'range ours (ln)': float(np.ptp(b_)),
                     'edf': GAM.loc[g, f'edf_{cond}']})
A2E = pd.DataFrame(rows)
A2E['decile'] = A2E.gene.map(dict(zip(GENES, decile)))
display(A2E.groupby('condition')[['r', 'RMS difference (ln)', 'range ours (ln)', 'edf']].median().round(3))
display(A2E.groupby('decile')[['r', 'RMS difference (ln)', 'edf']].median().round(3))
A2E.to_csv(tables / 'a2e_penalty_and_curves.csv', index=False)
EDF_SHARE = float((edf.stack() >= 5).mean())
print(f'Share of condition smooths with edf ≥ 5 (of 6): {EDF_SHARE:.3f}; median theta {GAM.theta.median():.2f}')

## 5 · B · NB diagnostics at unit level (gene subset G)

71's species M_full (designs, weights, knots, α) refitted for G gives μ; residuals and tests are unweighted.

In [ ]:
designs, W = contrast_designs(POSITION, PARTITIONS['species'][0], SPECIMEN, KNOTS)
XFULL = designs['full']
YG = COUNTS[:, SUB].toarray()
AG = ALPHA71[SUB]


def full_fit_g():
    f = nb_gam.nb_fit(YG, XFULL, W, np.log(LIBRARY), {'alpha': AG, 'converged': np.ones(len(SUB), bool)}, n_jobs=N_JOBS)
    return {'beta': f['beta'], 'converged': f['converged']}


FIT = cached('b_full_fit_subset', full_fit_g, inputs={**INPUT_KEY, 'subset': SUBSET, 'alpha': AG})
MU = np.clip(np.exp(XFULL @ FIT['beta'].T + np.log(LIBRARY)[:, None]), 1e-12, 1e8)
OKG = FIT['converged'].astype(bool) & M71['species']['stat|converged'][SUB].astype(bool) & ~M71['species']['stat|separated'][SUB].astype(bool)
P_FULL = np.linalg.matrix_rank(XFULL)
B = pd.DataFrame({'gene': SUBSET, 'decile': decile[SUB], 'alpha': AG, 'mean count': YG.mean(0), 'usable': OKG}).set_index('gene')

# B1 zeros
z = ga.zero_check(YG, MU, AG)
B['zeros observed'], B['zeros NB'], B['zeros Poisson'] = z.observed.to_numpy(), z.nb.to_numpy(), z.poisson.to_numpy()
B['zero q'] = np.nan
B.loc[B.usable, 'zero q'] = false_discovery_control(z.p.to_numpy()[OKG])
B['zero misfit'] = (B['zero q'] < .05) & (B['zeros observed'] - B['zeros NB'] >= MISFIT['zero'])
# B2 Pearson dispersion
pdisp = ga.pearson_dispersion(YG, MU, AG, P_FULL)
B['Pearson dispersion NB'], B['Pearson dispersion Poisson'] = pdisp.nb.to_numpy(), pdisp.poisson.to_numpy()
# B3 randomized quantile residuals
RQR = ga.quantile_residuals(YG, MU, AG, np.random.default_rng(SEED))
ks = ga.residual_ks(RQR)
B['KS D'], B['KS p'] = ks.D.to_numpy(), ks.p.to_numpy()
B['KS q'] = np.nan
B.loc[B.usable, 'KS q'] = false_discovery_control(ks.p.to_numpy()[OKG])
B['residual misfit'] = (B['KS q'] < .05) & (B['KS D'] >= MISFIT['ks'])


# B4 offset slope
def offset_slope():
    x = np.column_stack([XFULL, np.log(LIBRARY) - np.log(LIBRARY).mean()])
    disp = {'alpha': AG, 'converged': np.ones(len(SUB), bool)}
    lr = nb_gam.nested_nb_lr(YG, {'full': XFULL, 'lib': x}, W, np.log(LIBRARY), disp, {'LR': ('full', 'lib')}, n_jobs=N_JOBS)
    fit = nb_gam.nb_fit(YG, x, W, np.log(LIBRARY), disp, n_jobs=N_JOBS)
    return {'LR': lr['LR'], 'b': fit['beta'][:, -1], 'converged': lr['converged']}


OS = cached('b_offset_slope', offset_slope, inputs={**INPUT_KEY, 'subset': SUBSET, 'alpha': AG})
B['offset slope'] = 1 + OS['b']
B['slope q'] = np.nan
B.loc[B.usable, 'slope q'] = false_discovery_control(chi2.sf(OS['LR'], 1)[OKG])
B['slope misfit'] = (B['slope q'] < .05) & ((B['offset slope'] - 1).abs() >= MISFIT['slope'])


# B5–B7 dispersion by position third, unit size tertile, specimen
def within_specimen_bins(values, k):
    out = np.empty(len(values), object)
    for s in np.unique(SPECIMEN):
        rows = SPECIMEN == s
        ranks = pd.Series(values[rows]).rank(method='first').to_numpy()
        out[rows] = np.minimum((ranks - 1) * k // rows.sum(), k - 1).astype(int)
    return out.astype(int)


THIRD = within_specimen_bins(POSITION, 3)
SIZE = within_specimen_bins(N_SPOTS, 3)


def grouped():
    out = {}
    for name, groups in (('third', THIRD), ('size', SIZE), ('specimen', SPECIMEN)):
        per, test = ga.grouped_alpha(YG, MU, AG, groups)
        for c in per.columns:
            out[f'{name}|{c}'] = per[c].to_numpy()
        out[f'{name}|LR'], out[f'{name}|p'] = test.LR.to_numpy(), test.p.to_numpy()
        out[f'{name}|common'] = test.alpha_common.to_numpy()
    return out


GR = cached('b_grouped_alpha', grouped, inputs={**INPUT_KEY, 'subset': SUBSET, 'alpha': AG, 'third': THIRD, 'size': SIZE})
thirds = np.column_stack([GR[f'third|{k}'] for k in range(3)])
B['alpha thirds max ratio'] = thirds.max(1) / thirds.min(1)
B['third q'] = np.nan
B.loc[B.usable, 'third q'] = false_discovery_control(GR['third|p'][OKG])
B['third misfit'] = (B['third q'] < .05) & (B['alpha thirds max ratio'] >= MISFIT['alpha_ratio'])
sizes = np.column_stack([GR[f'size|{k}'] for k in range(3)])
med_spots = np.array([np.median(N_SPOTS[SIZE == k]) for k in range(3)])
lx = np.log(med_spots) - np.log(med_spots).mean()
B['alpha size slope'] = (np.log(sizes) - np.log(sizes).mean(1, keepdims=True)) @ lx / (lx @ lx)
B['size q'] = np.nan
B.loc[B.usable, 'size q'] = false_discovery_control(GR['size|p'][OKG])
B['size misfit'] = (B['size q'] < .05) & (sizes.max(1) / sizes.min(1) >= MISFIT['alpha_ratio'])
spec_alpha = pd.DataFrame({s: GR[f'specimen|{s}'] for s in sorted(set(SPECIMEN))}, index=SUBSET)
B['alpha human / mouse'] = np.exp(np.log(spec_alpha[['HUK1_COR1', 'HUK1_MED1']]).mean(1) - np.log(spec_alpha[['Ctrl1A2', 'Ctrl1A4']]).mean(1))
B.to_csv(tables / 'b_diagnostics_subset.csv')
U_ = B[B.usable]
B_SUMMARY = pd.Series({
    'genes (usable of G)': f'{len(U_)} of {len(B)}',
    'B1 zero misfit (q < 0.05 and excess ≥ 0.02)': U_['zero misfit'].mean(),
    'B1 median observed − NB expected zeros': (U_['zeros observed'] - U_['zeros NB']).median(),
    'B1 median observed − Poisson expected zeros': (U_['zeros observed'] - U_['zeros Poisson']).median(),
    'B2 median Pearson dispersion, NB': U_['Pearson dispersion NB'].median(),
    'B2 median Pearson dispersion, Poisson': U_['Pearson dispersion Poisson'].median(),
    'B3 residual misfit (q < 0.05 and KS D ≥ 0.05)': U_['residual misfit'].mean(),
    'B3 median KS D': U_['KS D'].median(),
    'B4 median offset slope': U_['offset slope'].median(),
    'B4 slope IQR': f"{U_['offset slope'].quantile(.25):.2f}–{U_['offset slope'].quantile(.75):.2f}",
    'B4 slope misfit (q < 0.05 and |slope − 1| ≥ 0.2)': U_['slope misfit'].mean(),
    'B5 α varies along the curve (q < 0.05 and ratio ≥ 2)': U_['third misfit'].mean(),
    'B5 median max/min α ratio of thirds': U_['alpha thirds max ratio'].median(),
    'B6 α varies with unit size (q < 0.05 and ratio ≥ 2)': U_['size misfit'].mean(),
    'B6 median slope of log α on log n_spots': U_['alpha size slope'].median(),
    'B7 median α human / mouse': U_['alpha human / mouse'].median()}, name='value')
display(B_SUMMARY.to_frame())
B_SUMMARY.to_csv(tables / 'b_summary.csv')
display(B[B.index.isin(MARKERS)][['alpha', 'mean count', 'zeros observed', 'zeros NB', 'Pearson dispersion NB', 'KS D',
                                    'offset slope', 'alpha thirds max ratio', 'alpha size slope']].round(3))

# B8 dispersion against mean over the universe, with the DESeq2 parametric trend
MEAN_ALL = np.asarray(COUNTS.mean(axis=0)).ravel()
TREND_A, TREND_B, TREND = ga.dispersion_trend(ALPHA71, MEAN_ALL)
SHRUNK, SHRINK_W = ga.shrunken_alpha(ALPHA71, TREND, len(POSITION), P_FULL)
print(f'B8 trend: α = {TREND_A:.4f} + {TREND_B:.4f} / mean; shrinkage weight on the gene estimate {SHRINK_W:.4f}; '
      f'Spearman(α, mean) {spearmanr(ALPHA71, MEAN_ALL).statistic:.3f}')

# Mean–variance, pooled over G: 20 bins of fitted μ
mv = pd.DataFrame({'mu': MU[:, OKG].ravel(), 'y': YG[:, OKG].ravel(),
                   'nb': (MU[:, OKG] + AG[OKG][None, :] * MU[:, OKG] ** 2).ravel()})
mv['bin'] = pd.qcut(np.log(mv.mu), 20, labels=False, duplicates='drop')
mv['sq'] = (mv.y - mv.mu) ** 2                # squared deviation from each unit's own fitted mean
MV = mv.groupby('bin').agg(mean=('mu', 'mean'), observed=('sq', 'mean'), nb=('nb', 'mean'), poisson=('mu', 'mean')).reset_index()
MV['observed / NB'] = MV.observed / MV.nb
MV['observed / Poisson'] = MV.observed / MV.poisson
display(MV.round(3))
MV.to_csv(tables / 'b2_mean_variance_pooled.csv', index=False)

## 6 · C1 · Basis df and knot placement by fit (gene subset G)

AIC and BIC from unweighted fits (α re-estimated by ML per df); 5-fold held-out deviance with 71's weights.

In [ ]:
FOLDS = ga.cv_folds(SPECIMEN, N_FOLDS, np.random.default_rng(SEED))


def design_for(knots):
    return contrast_designs(POSITION, PARTITIONS['species'][0], SPECIMEN, knots)[0]['full']


def c1_fit(knots):
    x = design_for(knots)
    disp = nb_gam.nb_dispersion(YG, x, np.ones(len(POSITION)), np.log(LIBRARY), n_jobs=N_JOBS)
    fit = nb_gam.nb_fit(YG, x, np.ones(len(POSITION)), np.log(LIBRARY), disp, n_jobs=N_JOBS)
    mu = np.clip(np.exp(x @ fit['beta'].T + np.log(LIBRARY)[:, None]), 1e-12, 1e8)
    ll = nb_gam._loglik(YG, mu, disp['alpha'], np.ones(len(POSITION)))
    k = np.linalg.matrix_rank(x) + 1
    dev, ok = ga.heldout_deviance(YG, x, W, np.log(LIBRARY), FOLDS, n_jobs=N_JOBS)
    return {'aic': -2 * ll + 2 * k, 'bic': -2 * ll + np.log(len(POSITION)) * k, 'cv': dev,
            'converged': fit['converged'] & disp['converged'] & ok, 'k': np.array(k)}


C1 = {}
for df in DFS:
    C1[f'df {df}'] = cached(f'c1_df{df}', lambda df=df: c1_fit(gam_internal_knots(POSITION, df)),
                            inputs={**INPUT_KEY, 'subset': SUBSET, 'folds': FOLDS}, params={'df': df})
EQUAL = np.array([.25, .5, .75])
C1['df 6, equal knots'] = cached('c1_equal_knots', lambda: c1_fit(EQUAL), inputs={**INPUT_KEY, 'subset': SUBSET, 'folds': FOLDS})
ok_all = np.logical_and.reduce([v['converged'].astype(bool) for v in C1.values()]) & OKG
rows = []
for name, v in C1.items():
    rows.append({'model': name, 'parameters': int(v['k']),
                 'median ΔAIC vs df 6': float(np.median((v['aic'] - C1['df 6']['aic'])[ok_all])),
                 'median ΔBIC vs df 6': float(np.median((v['bic'] - C1['df 6']['bic'])[ok_all])),
                 'median Δ held-out deviance vs df 6 (%)': float(np.median(100 * (v['cv'] - C1['df 6']['cv'])[ok_all] / C1['df 6']['cv'][ok_all])),
                 'sum Δ held-out deviance vs df 6': float(np.sum((v['cv'] - C1['df 6']['cv'])[ok_all])),
                 'genes better than df 6 (held-out)': float(np.mean((v['cv'] < C1['df 6']['cv'])[ok_all]))})
C1T = pd.DataFrame(rows).set_index('model')
best = {}
for crit in ('aic', 'bic', 'cv'):
    stack = np.column_stack([C1[f'df {d}'][crit] for d in DFS])[ok_all]
    best[crit] = pd.Series(np.array(DFS)[np.argmin(stack, axis=1)]).value_counts(normalize=True).reindex(DFS, fill_value=0)
C1BEST = pd.DataFrame(best).rename(columns={'aic': 'AIC best', 'bic': 'BIC best', 'cv': 'held-out best'})
display(C1T.round(3))
display(C1BEST.round(3))
C1T.to_csv(tables / 'c1_fit_by_df.csv')
C1BEST.to_csv(tables / 'c1_best_df_shares.csv')
BEST_CV = pd.Series(np.array(DFS)[np.argmin(np.column_stack([C1[f'df {d}']['cv'] for d in DFS]), axis=1)], index=SUBSET)[ok_all]
C1_BY_DECILE = BEST_CV.groupby(decile[SUB][ok_all]).agg(['median', 'mean'])
C1_BY_DECILE.index.name = 'decile of log mean count'
display(C1_BY_DECILE.T.round(2))
C1_BY_DECILE.to_csv(tables / 'c1_best_df_by_decile.csv')

## 7 · C2 · Do the choices change 71's results? (full universe)

Each variant reruns the positional F-test on all genes and partitions with one change. The baseline (71's settings)
must reproduce 71's T_spatial and LR_spec (guard G-C).

In [ ]:
def variant_run(knots, *, unit_weights=False, offset=None, alpha=None):
    out = {}
    disp = None if alpha is None else {'alpha': np.asarray(alpha, float), 'converged': np.ones(len(GENES), bool)}
    for label in LABELS:
        group, nuisance = PARTITIONS[label]
        r = ga.positional_lrs(COUNTS, LIBRARY, POSITION, group, SPECIMEN, knots, dispersion=disp, nuisance_shape=nuisance,
                              unit_weights=unit_weights, offset=offset, n_jobs=N_JOBS)
        if disp is None:
            disp = r['dispersion']                     # the species split's, reused for the relabelings (71)
        for k in ('T_spatial', 'T_total', 'LR_spec', 'converged', 'separated'):
            out[f'{SHORT[label]}|{k}'] = np.asarray(r[k])
        out[f'{SHORT[label]}|df2'] = np.array(r['df2'])
        out[f'{SHORT[label]}|df1'] = np.array(r['df1'])
    out['alpha'] = np.asarray(disp['alpha'], float)
    return out


offsets_r = pd.read_csv(ts_dir / 'offsets.csv').set_index('unit').reindex(OBS.index.astype(str))
assert offsets_r.notna().all().all()
OFF_TMM = offsets_r.offset_tmm.to_numpy(float)
SF = ga.poscounts_size_factors(COUNTS)
VARIANTS = {'71 (baseline)': dict(knots=KNOTS)}
for df in VARIANT_DFS:
    VARIANTS[f'df {df}'] = dict(knots=gam_internal_knots(POSITION, df))
VARIANTS.update({'equal knots': dict(knots=EQUAL), 'unit weights': dict(knots=KNOTS, unit_weights=True),
                 'trended α': dict(knots=KNOTS, alpha=TREND), 'shrunken α': dict(knots=KNOTS, alpha=SHRUNK),
                 'TMM offset': dict(knots=KNOTS, offset=OFF_TMM), 'poscounts offset': dict(knots=KNOTS, offset=np.log(SF) + np.log(LIBRARY).mean())})
RUNS = {}
for name, kw in VARIANTS.items():
    key = {k: (v if not isinstance(v, np.ndarray) else digest(v)) for k, v in kw.items()}
    RUNS[name] = cached(f"c2_{name.replace(' ', '_').replace('(', '').replace(')', '').replace('α', 'alpha')}",
                        lambda kw=kw: variant_run(**kw), inputs={**INPUT_KEY, **key})

base = RUNS['71 (baseline)']
GUARD_C = []
alpha_ratio = np.maximum(base['alpha'], ALPHA71) / np.minimum(base['alpha'], ALPHA71)
for label in LABELS:
    s = SHORT[label]
    for stat, ref, conv in (('T_spatial', M71[label]['stat|T_spatial'], M71[label]['stat|converged']),
                            ('LR_spec', D71[label]['LR_spec'], D71[label]['converged'])):
        ok = base[f'{s}|converged'].astype(bool) & conv.astype(bool)
        scaled = np.abs(base[f'{s}|{stat}'][ok] - ref[ok]) / np.maximum(np.abs(ref[ok]), 1.)
        GUARD_C.append({'partition': s, 'statistic': stat, 'genes': int(ok.sum()), 'max scaled difference': float(scaled.max()),
                        'genes beyond 1e-6': int((scaled > 1e-6).sum()), 'genes beyond 1e-4': int((scaled > 1e-4).sum()),
                        'share beyond 1e-4': float((scaled > 1e-4).mean())})
GUARD_C = pd.DataFrame(GUARD_C)
display(GUARD_C)
GUARD_C.to_csv(tables / 'guard_c_reproduction.csv', index=False)
print(f'α against 71: {int((alpha_ratio > 1 + 1e-6).sum())} genes differ beyond 1e-6 (largest ratio {alpha_ratio.max():.2f}, '
      f'median α of those {np.median(ALPHA71[alpha_ratio > 1 + 1e-6]):.2e})')
# Deviation (recorded): the protocol asked rtol 1e-6. A rerun of 71's code re-estimates α, which differs for a few genes on
# a flat likelihood, so a few LRs differ. The run stops when more than 0.5% of genes differ beyond 1e-4 (scaled by
# max(|value|, 1)); the run-to-run floor is reported in C2 as the baseline against 71's stored results.
assert GUARD_C['share beyond 1e-4'].max() <= .005, GUARD_C
GUARDS.append(f"G-C: the baseline reproduces 71's T_spatial and LR_spec for ≥ {1 - GUARD_C['share beyond 1e-4'].max():.4f} of genes "
              f"in every partition at 1e-4; {int(GUARD_C['genes beyond 1e-4'].sum())} gene-statistics differ more "
              f"(α re-estimated: {int((alpha_ratio > 1 + 1e-6).sum())} genes beyond 1e-6)")
print(GUARDS[-1])


def screen(run):
    stat, tests = {}, {}
    for label in LABELS:
        s = SHORT[label]
        frame = moderated_f(pd.Series(run[f'{s}|T_spatial'], index=GENES), int(run[f'{s}|df1']), run[f'{s}|LR_spec'],
                            int(run[f'{s}|df2']), covariate=LOG_MEAN, robust=False)
        frame['q'] = false_discovery_control(frame.p.to_numpy())
        stat[s] = frame
        tests[s] = camera_pr(frame.z, SETS, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
    return stat, tests


SCREENS = {name: screen(run) for name, run in RUNS.items()}
STORED = {}, {}
for r in ('condition', 'relabeling 1', 'relabeling 2'):
    frame = S71[r][['p', 'z', 'q']].copy()
    STORED[0][r] = frame
    STORED[1][r] = camera_pr(frame.z, SETS, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
SCREENS = {'71 as stored': STORED, **SCREENS}
stat0, tests0 = SCREENS['71 (baseline)']
Z_DIFF = float(np.max(np.abs(stat0['condition'].z.to_numpy() - S71[('condition', 'z')].to_numpy())))
print(f'Baseline condition z against 71 stored: max absolute difference {Z_DIFF:.2e}, '
      f"Spearman {spearmanr(stat0['condition'].z, S71[('condition', 'z')]).statistic:.6f}")
calls0 = set(tests0['condition'].pathway_id[tests0['condition'].call.astype(bool)])
genes0 = set(stat0['condition'].index[stat0['condition'].q <= ALPHA])
rows = []
for name, (stat, tests) in SCREENS.items():
    calls = set(tests['condition'].pathway_id[tests['condition'].call.astype(bool)])
    rel = [int(tests[r].call.astype(bool).sum()) for r in ('relabeling 1', 'relabeling 2')]
    p_rel = np.concatenate([tests[r].p_one_sided.to_numpy(float) for r in ('relabeling 1', 'relabeling 2')])
    g_rel = np.concatenate([stat[r].p.to_numpy(float) for r in ('relabeling 1', 'relabeling 2')])
    genes = set(stat['condition'].index[stat['condition'].q <= ALPHA])
    ncdr = np.mean(rel) / len(calls) if calls else np.nan
    share = float(np.mean(p_rel <= ALPHA))
    rows.append({'variant': name, 'z Spearman vs 71': spearmanr(stat['condition'].z, stat0['condition'].z).statistic,
                 'genes q ≤ 0.05': len(genes), 'gene Jaccard vs 71': len(genes & genes0) / max(len(genes | genes0), 1),
                 'gene-level relabeled share': float(np.mean(g_rel <= ALPHA)),
                 'pathway calls': len(calls), 'relabeled calls': f'{rel[0]}, {rel[1]}', 'NCDR': ncdr,
                 'calibration share': share, 'call Jaccard vs 71': len(calls & calls0) / max(len(calls | calls0), 1),
                 'passes 71\'s rule': bool(calls) and ncdr < NCDR_MAX and max(rel) < NULL_RATE * len(SETS) and share <= CALIBRATION_MAX})
C2 = pd.DataFrame(rows).set_index('variant')
display(C2.round(3))
C2.to_csv(tables / 'c2_variants.csv')

## 7b · Post hoc checks (added after the first complete run; recorded as deviations)

- **PH1 · Which genes want fewer basis functions?** C1's held-out best df, split by whether 71's positional F-test calls
  the gene (species q ≤ 0.05).
- **PH2 · Why the offset matters.** tradeSeq's TMM factor (log(TMM offset) − log(its library)) along the curve, by
  species: a factor that trends with position in one species enters every gene's species × position term.

In [ ]:
called71 = (S71[('condition', 'q')] <= ALPHA).reindex(SUBSET).to_numpy()
PH1 = pd.DataFrame({'held-out best df': BEST_CV, 'called by 71': called71[ok_all]})
PH1_TABLE = PH1.groupby('called by 71')['held-out best df'].agg(['count', 'median', 'mean', lambda v: float((v >= 6).mean())])
PH1_TABLE.columns = ['genes', 'median best df', 'mean best df', 'share best at df ≥ 6']
display(PH1_TABLE.round(3))
PH1_TABLE.to_csv(tables / 'ph1_best_df_by_call.csv')
DELTA_CALLED = pd.DataFrame({f'df {d}': (C1[f'df {d}']['cv'] - C1['df 6']['cv'])[ok_all] for d in DFS}, index=np.array(SUBSET)[ok_all])
display(DELTA_CALLED.groupby(called71[ok_all]).median().round(2))
DELTA_CALLED.groupby(called71[ok_all]).median().to_csv(tables / 'ph1_heldout_change_by_call.csv')

nf = pd.read_csv(ts_dir / 'offsets.csv').set_index('unit').reindex(OBS.index.astype(str))
log_nf = nf.offset_tmm.to_numpy(float) - np.log(nf.library.to_numpy(float))
deciles = pd.qcut(POSITION, 10, labels=False)
PH2 = pd.DataFrame({'species': np.where(HUMAN, 'human', 'mouse'), 'decile': deciles, 'log TMM factor': log_nf}) \
    .groupby(['species', 'decile'])['log TMM factor'].mean().unstack(0)
display(PH2.round(3).T)
PH2.to_csv(tables / 'ph2_tmm_factor_along_curve.csv')
PH2_RHO = {sp: float(spearmanr(log_nf[HUMAN == (sp == 'human')], POSITION[HUMAN == (sp == 'human')]).statistic) for sp in ('human', 'mouse')}
print('Spearman of the log TMM factor with position:', {k: round(v, 3) for k, v in PH2_RHO.items()})

## 8 · Figures

- **87a** tradeSeq against ours: relabeled shares and calls; −log10 p (species) of tradeSeq against the raw T_total LR
  and the positional F-test; edf of tradeSeq's smooths; example curves.
- **87b** NB diagnostics.
- **87c** parameters.

In [ ]:
def figure_a(stem):
    fig = plt.figure(figsize=(180 * MM, 105 * MM))
    gs = fig.add_gridspec(2, 4, wspace=.45, hspace=.55)
    ax = fig.add_subplot(gs[0, :2])
    order = list(A2BC.index)
    colors = [C_OURS if m.startswith('ours') else (C_TSU if 'U' in m else C_TS) for m in order]
    ax.barh(range(len(order)), A2BC['relabeled share p ≤ 0.05'], color=colors)
    ax.axvline(.05, color=GREY, lw=.6, ls=':')
    ax.axvline(CALIBRATION_MAX, color=INK, lw=.6)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order)
    ax.invert_yaxis()
    ax.set_xlabel('relabeled genes with p ≤ 0.05')
    ax = fig.add_subplot(gs[0, 2:])
    sp = A2BC['species calls (BH ≤ 0.05)']
    rl = A2BC['relabeled calls'].str.split(', ', expand=True).astype(int).mean(1)
    ax.scatter(sp, rl, c=colors, s=14)
    for m, x, y in zip(order, sp, rl):
        ax.annotate(m.replace('ours: ', '').replace(' (χ², 7 df)', '').replace(' (χ², 6 df)', ''), (x, y), fontsize=5,
                    xytext=(5, 7 if 'T_spatial' in m else 0), textcoords='offset points', ha='left', va='center')
    lim = max(sp.max(), rl.max()) * 1.05
    ax.set_xlim(0, sp.max() * 1.45)
    ax.plot([0, lim], [0, lim], color=LIGHT, lw=.6)
    ax.plot([0, lim], [0, NCDR_MAX * lim], color=GREY, lw=.6, ls=':')
    ax.set_xlabel('species genes (BH ≤ 0.05)')
    ax.set_ylabel('relabeled genes (mean)')
    for j, (method, label) in enumerate((('ours: T_total LR (χ², 7 df)', 'raw T_total'), ('ours: positional F-test', 'positional F'))):
        ax = fig.add_subplot(gs[1, j])
        x = -np.log10(P[('tradeSeq conditionTest', 'condition')].clip(lower=1e-300))
        y = -np.log10(P[(method, 'condition')].clip(lower=1e-300))
        ok = x.notna() & y.notna()
        ax.hexbin(x[ok], y[ok], gridsize=40, bins='log', cmap='Greys', mincnt=1, lw=0, xscale='log', yscale='log')
        ax.set_xlabel('tradeSeq −log10 p')
        ax.set_ylabel(f'{label} −log10 p')
    ax = fig.add_subplot(gs[1, 2])
    ax.hist(edf.stack(), bins=np.linspace(1, 6, 26), color=C_TS)
    ax.axvline(6, color=INK, lw=.6)
    ax.set_xlabel('tradeSeq edf per smooth')
    ax.set_ylabel('smooths')
    ax = fig.add_subplot(gs[1, 3])
    for g, color in zip([m for m in ('Slc34a1', 'Slc7a13', 'Gpx3') if m in GAM.index], ('#0072B2', '#009E73', '#CC79A7')):
        i = idx71[g]
        for cond, key, ls in (('human', 'case', '-'), ('mouse', 'reference', '--')):
            tsv = CURVES[(CURVES.gene == g) & (CURVES.condition == cond)].sort_values('point').value.to_numpy()
            ours = M71['species'][key][i]
            ax.plot(GRID, (ours - ours.mean()) / np.log(2), color=color, ls=ls, lw=1.2)
            ax.plot(GRID, (tsv - tsv.mean()) / np.log(2), color=color, ls=ls, lw=.5, alpha=.9)
        ax.annotate(g, (GRID[-1], (ours[-1] - ours.mean()) / np.log(2)), fontsize=5, color=color)
    ax.set_xlabel('elastic curve')
    ax.set_ylabel('log2, centred')
    save(fig, stem)


figure_a('fig87a_tradeseq')


def figure_b(stem):
    U_ = B[B.usable]
    fig, axes = plt.subplots(2, 4, figsize=(180 * MM, 90 * MM), gridspec_kw={'wspace': .45, 'hspace': .55})
    ax = axes[0, 0]
    ax.scatter(U_['zeros NB'], U_['zeros observed'], s=2, color=C_OURS, label='NB', lw=0)
    ax.scatter(U_['zeros Poisson'], U_['zeros observed'], s=2, color=GREY, label='Poisson', lw=0)
    ax.plot([0, 1], [0, 1], color=INK, lw=.5)
    ax.set_xlabel('expected zero fraction')
    ax.set_ylabel('observed zero fraction')
    ax.legend(markerscale=3)
    ax = axes[0, 1]
    ax.semilogx(MV['mean'], MV['observed / NB'], 'o-', ms=2.5, color=C_OURS, label='NB')
    ax.semilogx(MV['mean'], MV['observed / Poisson'], 'o-', ms=2.5, color=GREY, label='Poisson')
    ax.axhline(1, color=INK, lw=.6)
    ax.set_xlabel('fitted mean (bin)')
    ax.set_ylabel('observed / expected variance')
    ax.legend()
    ax = axes[0, 2]
    keep = np.flatnonzero(OKG)
    pooled = np.sort(np.random.default_rng(SEED).choice(RQR[:, keep].ravel(), 20000, replace=False))
    theo = np.sort(np.random.default_rng(SEED + 1).standard_normal(20000))
    ax.plot(theo, pooled, '.', ms=1, color=C_OURS)
    ax.plot([-4, 4], [-4, 4], color=INK, lw=.5)
    ax.set_xlabel('N(0, 1) quantile')
    ax.set_ylabel('quantile residual')
    ax = axes[0, 3]
    ax.hist(U_['Pearson dispersion NB'].clip(upper=3), bins=40, color=C_OURS)
    ax.axvline(1, color=INK, lw=.6)
    ax.set_xlabel('Pearson dispersion (NB)')
    ax.set_ylabel('genes')
    ax = axes[1, 0]
    ax.hist(U_['offset slope'].clip(0, 2), bins=40, color=C_OURS)
    ax.axvline(1, color=INK, lw=.6)
    ax.set_xlabel('library slope')
    ax.set_ylabel('genes')
    ax = axes[1, 1]
    th = np.log2(thirds[OKG] / thirds[OKG].mean(1, keepdims=True))
    ax.boxplot([th[:, k] for k in range(3)], showfliers=False, widths=.5, medianprops={'color': C_OURS})
    ax.axhline(0, color=GREY, lw=.5)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(['early', 'middle', 'late'])
    ax.set_ylabel('log2 α / mean')
    ax = axes[1, 2]
    sz = np.log2(sizes[OKG] / sizes[OKG].mean(1, keepdims=True))
    ax.boxplot([sz[:, k] for k in range(3)], showfliers=False, widths=.5, medianprops={'color': C_OURS})
    ax.axhline(0, color=GREY, lw=.5)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([f'{m:.0f}' for m in med_spots])
    ax.set_xlabel('unit size (spots)')
    ax.set_ylabel('log2 α / mean')
    ax = axes[1, 3]
    ax.loglog(MEAN_ALL, ALPHA71, '.', ms=.6, color=GREY)
    o = np.argsort(MEAN_ALL)
    ax.loglog(MEAN_ALL[o], TREND[o], color=INK, lw=.8)
    ax.set_xlabel('mean count')
    ax.set_ylabel('α')
    save(fig, stem)


figure_b('fig87b_nb_diagnostics')


def figure_c(stem):
    fig, axes = plt.subplots(1, 3, figsize=(180 * MM, 52 * MM), gridspec_kw={'wspace': .45})
    ax = axes[0]
    for crit, color, label in (('aic', C_OURS, 'AIC'), ('bic', GREY, 'BIC'), ('cv', INK, 'held-out deviance')):
        med = [np.median((C1[f'df {d}'][crit] - C1['df 6'][crit])[ok_all]) for d in DFS]
        ax.plot(DFS, med, 'o-', color=color, ms=3, label=label)
    ax.axhline(0, color=LIGHT, lw=.6)
    ax.axvline(6, color=LIGHT, lw=.6)
    ax.set_xlabel('basis df')
    ax.set_ylabel('median change vs df 6')
    ax.legend()
    ax = axes[1]
    width = .25
    for j, (col, color) in enumerate((('AIC best', C_OURS), ('BIC best', GREY), ('held-out best', INK))):
        ax.bar(np.arange(len(DFS)) + (j - 1) * width, C1BEST[col], width=width, color=color)
    ax.set_xticks(range(len(DFS)))
    ax.set_xticklabels(DFS)
    ax.set_xlabel('basis df')
    ax.set_ylabel('share of genes best')
    ax = axes[2]
    v = C2.drop(index=['71 (baseline)', '71 as stored'])
    ax.scatter(v['call Jaccard vs 71'], v['calibration share'], color=[C_OURS if r else '#D55E00' for r in v["passes 71's rule"]], s=14)
    for name, r in v.iterrows():
        ax.annotate(name, (r['call Jaccard vs 71'], r['calibration share']), fontsize=5, xytext=(3, 2), textcoords='offset points')
    ax.axhline(CALIBRATION_MAX, color=INK, lw=.6)
    ax.set_xlabel('pathway calls shared with 71 (Jaccard)')
    ax.set_ylabel('calibration share')
    save(fig, stem)


figure_c('fig87c_parameters')

## 9 · Outputs and run record

In [ ]:
import platform
from importlib.metadata import version

record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-gene-model-audit.md (0c8d28a)',
          'guards': GUARDS, 'guard_c': GUARD_C.to_dict(orient='records'), 'z_difference': Z_DIFF, 'subset': SUBSET, 'tradeseq': {'key': TS_KEY, 'seconds': TS_SECONDS,
                                                            'session': (ts_dir / 'session.txt').read_text().splitlines()[-3:]},
          'a2bc': A2BC.reset_index().to_dict(orient='records'), 'a2a': A2A.reset_index().to_dict(orient='records'),
          'a2d': A2D.to_dict(), 'edf_share_ge5': EDF_SHARE, 'b_summary': B_SUMMARY.astype(str).to_dict(),
          'trend': {'a': TREND_A, 'b': TREND_B, 'shrink_weight': SHRINK_W},
          'c1': C1T.reset_index().to_dict(orient='records'), 'c1_best': C1BEST.reset_index().to_dict(orient='records'),
          'c2': C2.reset_index().to_dict(orient='records'), 'ph1': PH1_TABLE.reset_index().to_dict(orient='records'),
          'ph2_rho': PH2_RHO,
          'versions': {'python': platform.python_version(), **{m: version(m) for m in ('numpy', 'scipy', 'pandas')}},
          'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds'] / 60:.1f} min")